# YOLOv8 Pose Detection on Videos

## Introduction

This script uses the YOLOv8 model to perform pose detection on videos, extract keypoints for the lowest player, and save the results in CSV files.

### Setting Up the Environment

First, we set up the environment by cloning the Ultralytics repository and setting paths for model weights and video sources.

```python

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

# Clone the Ultralytics repository
%cd {HOME}
!git clone https://github.com/ultralytics/ultralytics.git
%cd {HOME}/ultralytics

# Add the Ultralytics directory to the Python path
import sys
sys.path.append(f"{HOME}/ultralytics")

# Download the model weights
%cd {HOME}/ultralytics
!wget /content/drive/MyDrive/Penalty_project/yolov8x-pose.pt --quiet

# Define the path for the model weights
POSE_MODEL_WEIGHTS_PATH = f"{HOME}/drive/MyDrive/Penalty_project/yolov8x-pose.pt"

### Importing Libraries

We import the necessary libraries for model loading and data handling.



In [ ]:
import torch
from ultralytics import YOLO
import pandas as pd

# Set the device for computation
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the YOLO model
model = YOLO('yolov8x-pose.pt')


Using the model on one video (for testing)

In [ ]:
# Path to the video
video_path = "/content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_76.mp4"
video_name = os.path.basename(video_path)

# Executing the model on the video
results = model(source=video_path, show=True, conf=0.03, save=True, augment=False, max_det=3, retina_masks=True)


### Defining the Detection Function
This function processes a video, detects poses, and extracts keypoints for the lowest player in each frame.

In [1]:
def f(video_path, output_dir):
    video_name = os.path.basename(video_path)

    # Run the model on the video
    results = model(source=video_path, show=False, conf=0.15, save=False, augment=False, max_det=4, retina_masks=False)

    # Define the output directory based on the video name
    output_dir = os.path.join(output_dir, os.path.splitext(video_name)[0])
    os.makedirs(output_dir, exist_ok=True)

    # Filter to get only the lowest player
    for index, result in enumerate(results):
        if result.boxes.data.shape[0] > 0:  # Ensure there are detected boxes
            y_max = result.boxes.data[:, 3]  # Get all y2 coordinates
            lowest_box_index = torch.argmax(y_max).item()  # Get the index of the lowest box
        else:
            print(f"No boxes detected in frame {index}.")

        # For each keypoint
        if result.keypoints and len(result.keypoints) > lowest_box_index:
            keypoints = result.keypoints[lowest_box_index]
            keypoints_data = []
            for i in range(5, 17):
                keypoints_data.append({
                    'x': keypoints.xy[0][i][0].numpy(),  # Convert tensor to numpy array
                    'y': keypoints.xy[0][i][1].numpy(),
                    'z': keypoints.conf[0][i].numpy(),
                })

            # Create a DataFrame from keypoints data
            df_keypoints = pd.DataFrame(keypoints_data)

            # Generate a unique filename for the CSV
            csv_filename = f"keypoints_frame_{index}.csv"
            csv_file = os.path.join(output_dir, csv_filename)

            # Export to CSV
            df_keypoints.to_csv(csv_file, index=True)
        else:
            print(f"No keypoints found for frame {index} or incorrect index reference.")


### Processing the Video Files
This section processes each video file in the specified directory, runs the detection function, and saves the results.

In [ ]:
# Paths to the dataset directories
input_directory = '/content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25'
output_directory = '/content/drive/MyDrive/Penalty_project/My_Dataset/output_2'

for file in os.listdir(input_directory):
    if file.endswith('.mp4'):
        output_folder_name = file[:-4]  # Remove the .mp4 extension to get the folder name
        full_path = os.path.join(input_directory, file)
        output_path = os.path.join(output_directory, output_folder_name)

        if not os.path.exists(output_path):  # Check if the output file already exists
            print(f'Processing video: {full_path}')
            f(full_path, output_directory)
        else:
            print(f'The video {full_path} has already been processed.')


## Conclusion
This script uses the YOLOv8 model to detect poses in videos, extracts keypoints for the lowest player, and saves the results in CSV files for further analysis.